In [1]:
import torch
import numpy as np
import json
import os
import argparse
from environment import ReKepOGEnv  # 环境封装（基于 OmniGibson）
from keypoint_proposal import KeypointProposer  # 关键点提议模块（由 VLM/图像处理得到）
from constraint_generation import ConstraintGenerator  # 将文字任务转为多阶段约束
from ik_solver import IKSolver  # 逆运动学求解器（可达性、姿态约束等）
from subgoal_solver import SubgoalSolver  # 子目标姿态优化器（满足子目标与路径约束）
from path_solver import PathSolver  # 轨迹优化器（从当前到子目标的平滑路径）
from visualizer import Visualizer  # 可视化工具（调试/演示）
import transform_utils as T
from omnigibson.robots.fetch import Fetch  # 本实现基于 Fetch 机械臂
from utils import (
    bcolors,
    get_config,
    load_functions_from_txt,
    get_linear_interpolation_steps,
    spline_interpolate_poses,
    get_callable_grasping_cost_fn,
    print_opt_debug_dict,
)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['OPENAI_API_KEY'] = "sk-Tlcxq05dMBH8wGU5S6SB1ujN20TviKkjhwyoV3orQ2j76IDF"
os.environ['OPENAI_API_BASE_URL'] = "http://chatapi-hk.littlewheat.com/v1"

In [3]:
parser = argparse.ArgumentParser()
parser.add_argument('--task', type=str, default='pen', help='要执行的任务')
parser.add_argument('--use_cached_query', action='store_true', help='使用缓存的查询而不是查询VLM')
parser.add_argument('--apply_disturbance', action='store_true', help='应用干扰以测试鲁棒性')
parser.add_argument('--visualize', action='store_true', help='在执行前可视化每个解决方案（注意：这会阻塞并需要按"ESC"继续）')
args, unknown = parser.parse_known_args()

In [4]:
args.use_cached_query=True # 适配默认的运行方式：python main.py --use_cached_query
args

Namespace(task='pen', use_cached_query=True, apply_disturbance=False, visualize=False)

In [5]:

if args.apply_disturbance: # 仅任务“pen”有缓存，在目录：“vlm_query/pen”
    assert args.task == 'pen' and args.use_cached_query, '干扰序列仅为缓存场景定义'

# 笔任务干扰序列
def stage1_disturbance_seq(env):
    """
    在机器人尝试抓取笔时移动笔的位置
    """
    pen = env.og_env.scene.object_registry("name", "pen_1")
    holder = env.og_env.scene.object_registry("name", "pencil_holder_1")
    # 干扰序列
    pos0, orn0 = pen.get_position_orientation()
    pose0 = np.concatenate([pos0, orn0])
    pos1 = pos0 + np.array([-0.08, 0.0, 0.0])  # 先左移
    orn1 = T.quat_multiply(T.euler2quat(np.array([0, 0, np.pi/4])), orn0)
    pose1 = np.concatenate([pos1, orn1])
    pos2 = pos1 + np.array([0.10, 0.0, 0.0])  # 再右移并改变姿态
    orn2 = T.quat_multiply(T.euler2quat(np.array([0, 0, -np.pi/2])), orn1)
    pose2 = np.concatenate([pos2, orn2])
    control_points = np.array([pose0, pose1, pose2])
    pose_seq = spline_interpolate_poses(control_points, num_steps=25)  # 生成连续移动轨迹
    def disturbance(counter):
        if counter < len(pose_seq):
            pose = pose_seq[counter]
            pos, orn = pose[:3], pose[3:]
            pen.set_position_orientation(pos, orn)
            counter += 1
    counter = 0
    while True:
        yield disturbance(counter)
        counter += 1

def stage2_disturbance_seq(env):
    """
    在机器人尝试重新定向笔时从夹爪中取出笔
    """
    apply_disturbance = env.is_grasping()  # 仅在抓住时才触发夺取
    pen = env.og_env.scene.object_registry("name", "pen_1")
    holder = env.og_env.scene.object_registry("name", "pencil_holder_1")
    # 干扰序列
    pos0, orn0 = pen.get_position_orientation()
    pose0 = np.concatenate([pos0, orn0])
    pose1 = np.array([-0.30, -0.15, 0.71, -0.7071068, 0, 0, 0.7071068])
    control_points = np.array([pose0, pose1])
    pose_seq = spline_interpolate_poses(control_points, num_steps=25)  # 将笔带走到远处
    def disturbance(counter):
        if apply_disturbance:
            if counter < 20:
                if counter > 15:
                    env.robot.release_grasp_immediately()  # 强制机器人释放笔
                else:
                    pass  # 其他步骤不做任何事
            elif counter < len(pose_seq) + 20:
                env.robot.release_grasp_immediately()  # 强制机器人释放笔
                pose = pose_seq[counter - 20]
                pos, orn = pose[:3], pose[3:]
                pen.set_position_orientation(pos, orn)
                counter += 1
    counter = 0
    while True:
        yield disturbance(counter)
        counter += 1

def stage3_disturbance_seq(env):
    """
    在机器人尝试将笔放入笔筒时移动笔筒
    """
    pen = env.og_env.scene.object_registry("name", "pen_1")
    holder = env.og_env.scene.object_registry("name", "pencil_holder_1")
    # 干扰序列
    pos0, orn0 = holder.get_position_orientation()
    pose0 = np.concatenate([pos0, orn0])
    pos1 = pos0 + np.array([-0.02, -0.15, 0.0])  # 轻微移动笔筒位置
    orn1 = orn0
    pose1 = np.concatenate([pos1, orn1])
    control_points = np.array([pose0, pose1])
    pose_seq = spline_interpolate_poses(control_points, num_steps=5)  # 缓慢移动
    def disturbance(counter):
        if counter < len(pose_seq):
            pose = pose_seq[counter]
            pos, orn = pose[:3], pose[3:]
            holder.set_position_orientation(pos, orn)
            counter += 1
    counter = 0
    while True:
        yield disturbance(counter)
        counter += 1

In [6]:

task_list = {
    'pen': {
        'scene_file': './configs/og_scene_file_red_pen.json',
        'instruction': 'pick  the red pen and drop it upright into the black pen holder',
        'rekep_program_dir': None, # './vlm_query/pen',  # 使用缓存的 ReKep 程序时的目录
        'disturbance_seq': {1: stage1_disturbance_seq, 2: stage2_disturbance_seq, 3: stage3_disturbance_seq},
        },
}
task = task_list['pen']
scene_file = task['scene_file']
instruction = task['instruction']

In [7]:
# === 模块与求解器初始化、优化管线函数 ===
# 本单元负责：
# 1) 构建全局上下文 CTX（系统配置、边界、求解器实例等）
# 2) 定义高层接口函数，用于：
#    - 生成下一步子目标位姿(get_next_subgoal)
#    - 基于子目标规划路径(get_next_path)
#    - 将稀疏控制点处理成稠密轨迹(process_path)
#    - 阶段切换与关键点可动性更新(update_stage / update_keypoint_movable_mask)
#    - 执行抓取/释放动作(execute_grasp_action / execute_release_action)
# 3) 这些函数被主循环调用，组成“感知(关键点) → 约束求解(子目标/路径) → 执行动作”的闭环。

import os
import json
import numpy as np
import torch

# -----------------------
# 全局上下文
# -----------------------
CTX = {}

global_config = get_config(config_path="./configs/config.yaml")
CTX['config'] = global_config['main']
CTX['bounds_min'] = np.array(CTX['config']['bounds_min'])
CTX['bounds_max'] = np.array(CTX['config']['bounds_max'])
o_constraint_generator =ConstraintGenerator(global_config['constraint_generator'])
o_KeypointProposer = KeypointProposer(global_config['keypoint_proposer'])
o_env = ReKepOGEnv(global_config['env'], task['scene_file'], verbose=False)


def init(scene_file, visualize=False):
    """
    初始化环境和各种求解器
    """
    global CTX
    
    CTX['visualize'] = visualize
    # 随机种子
    np.random.seed(CTX['config']['seed'])
    torch.manual_seed(CTX['config']['seed'])
    torch.cuda.manual_seed(CTX['config']['seed'])

    # 模块
    

    assert isinstance(o_env.robot, Fetch), "The IK solver assumes the robot is a Fetch robot"
    ik_solver = IKSolver(
        robot_description_path=o_env.robot.robot_arm_descriptor_yamls[o_env.robot.default_arm],
        robot_urdf_path=o_env.robot.urdf_path,
        eef_name=o_env.robot.eef_link_names[o_env.robot.default_arm],
        reset_joint_pos=o_env.reset_joint_pos,
        world2robot_homo=o_env.world2robot_homo,
    )

    CTX['subgoal_solver'] = SubgoalSolver(global_config['subgoal_solver'], ik_solver, o_env.reset_joint_pos)
    CTX['path_solver'] = PathSolver(global_config['path_solver'], ik_solver, o_env.reset_joint_pos)

    if visualize:
        CTX['visualizer'] = Visualizer(global_config['visualizer'], o_env)


def update_disturbance_seq(stage, disturbance_seq):
    if disturbance_seq is not None:
        if stage in disturbance_seq and not CTX['applied_disturbance'][stage]:
            o_env.disturbance_seq = disturbance_seq[stage](o_env)
            CTX['applied_disturbance'][stage] = True


def get_next_subgoal(from_scratch):
    subgoal_constraints = CTX['constraint_fns'][CTX['stage']]['subgoal']
    path_constraints = CTX['constraint_fns'][CTX['stage']]['path']
    subgoal_pose, debug_dict = CTX['subgoal_solver'].solve(
        CTX['curr_ee_pose'],
        CTX['keypoints'],
        CTX['keypoint_movable_mask'],
        subgoal_constraints,
        path_constraints,
        CTX['sdf_voxels'],
        CTX['collision_points'],
        CTX['is_grasp_stage'],
        CTX['curr_joint_pos'],
        from_scratch=from_scratch
    )
    subgoal_pose_homo = T.convert_pose_quat2mat(subgoal_pose)
    if CTX['is_grasp_stage']:
        subgoal_pose[:3] += subgoal_pose_homo[:3, :3] @ np.array([-CTX['config']['grasp_depth'] / 2.0, 0, 0])
    debug_dict['stage'] = CTX['stage']
    print_opt_debug_dict(debug_dict)
    if CTX['visualize']:
        CTX['visualizer'].visualize_subgoal(subgoal_pose)
    return subgoal_pose


def get_next_path(next_subgoal, from_scratch):
    path_constraints = CTX['constraint_fns'][CTX['stage']]['path']
    path, debug_dict = CTX['path_solver'].solve(
        CTX['curr_ee_pose'],
        next_subgoal,
        CTX['keypoints'],
        CTX['keypoint_movable_mask'],
        path_constraints,
        CTX['sdf_voxels'],
        CTX['collision_points'],
        CTX['curr_joint_pos'],
        from_scratch=from_scratch
    )
    print_opt_debug_dict(debug_dict)
    processed_path = process_path(path)
    if CTX['visualize']:
        CTX['visualizer'].visualize_path(processed_path)
    return processed_path


def process_path(path):
    full_control_points = np.concatenate([
        CTX['curr_ee_pose'].reshape(1, -1),
        path,
    ], axis=0)
    num_steps = get_linear_interpolation_steps(
        full_control_points[0], full_control_points[-1],
        CTX['config']['interpolate_pos_step_size'],
        CTX['config']['interpolate_rot_step_size']
    )
    dense_path = spline_interpolate_poses(full_control_points, num_steps)
    ee_action_seq = np.zeros((dense_path.shape[0], 8))
    ee_action_seq[:, :7] = dense_path
    ee_action_seq[:, 7] = o_env.get_gripper_null_action()
    return ee_action_seq


def update_stage(stage):
    CTX['stage'] = stage
    CTX['is_grasp_stage'] = CTX['program_info']['grasp_keypoints'][stage - 1] != -1
    CTX['is_release_stage'] = CTX['program_info']['release_keypoints'][stage - 1] != -1
    assert CTX['is_grasp_stage'] + CTX['is_release_stage'] <= 1
    if CTX['is_grasp_stage']:
        o_env.open_gripper()
    CTX['action_queue'] = []
    update_keypoint_movable_mask()
    CTX['first_iter'] = True


def update_keypoint_movable_mask():
    for i in range(1, len(CTX['keypoint_movable_mask'])):
        keypoint_object = o_env.get_object_by_keypoint(i - 1)
        CTX['keypoint_movable_mask'][i] = o_env.is_grasping(keypoint_object)


def execute_grasp_action():
    pregrasp_pose = o_env.get_ee_pose()
    grasp_pose = pregrasp_pose.copy()
    grasp_pose[:3] += T.quat2mat(pregrasp_pose[3:]) @ np.array([CTX['config']['grasp_depth'], 0, 0])
    grasp_action = np.concatenate([grasp_pose, [o_env.get_gripper_close_action()]])
    o_env.execute_action(grasp_action, precise=True)


def execute_release_action():
    o_env.open_gripper()


Using cache found in /home/youlika/.cache/torch/hub/facebookresearch_dinov2_main
/home/youlika/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/youlika/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/youlika/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
[INFO] [omnigibson.simulator] ----- Starting OmniGibson. This will take 10-30 seconds... -----


--------------------------------
{'scene': {'name': 'Rs_int', 'type': 'InteractiveTraversableScene', 'scene_model': 'Rs_int', 'scene_file': './configs/og_scene_file_red_pen.json'}, 'robots': [{'name': 'Fetch', 'type': 'Fetch', 'obs_modalities': ['rgb', 'depth'], 'action_modalities': 'continuous', 'action_normalize': False, 'position': [-0.8, 0.0, 0.0], 'grasping_mode': 'assisted', 'controller_config': {'base': {'name': 'DifferentialDriveController'}, 'arm_0': {'name': 'OperationalSpaceController', 'kp': 250, 'kp_limits': [50, 400], 'damping_ratio': 0.6}, 'gripper_0': {'name': 'MultiFingerGripperController', 'command_input_limits': [0.0, 1.0], 'mode': 'smooth'}, 'camera': {'name': 'JointController'}}}], 'env': {'physics_frequency': 60, 'action_frequency': 15}}
[Warning] [omni.isaac.kit] Interactive python shell detected but ISAAC_JUPYTER_KERNEL was not set. Problems with asyncio may occur
[Warning] [omni.isaac.kit] Please use Isaac Sim Python 3 kernel instead of the default Python 3 Ker

[INFO] [omni.kit.telemetry.impl.sentry_extension] sentry is disabled for external build
[INFO] [omni.kit.telemetry.impl.sentry_extension] sentry is disabled for external build


[0.881s] [ext: omni.kit.telemetry-0.5.0] startup
[0.911s] [ext: omni.kit.loop-isaac-1.2.0] startup
[0.912s] [ext: omni.kit.test-0.0.0] startup
[0.935s] [ext: omni.appwindow-1.1.8] startup
[1.049s] [ext: omni.kit.renderer.core-1.0.1] startup
[2.326s] [ext: omni.kit.renderer.capture-0.0.0] startup
[2.327s] [ext: omni.kit.renderer.imgui-1.0.1] startup
[2.392s] [ext: omni.ui-2.23.11] startup
[2.398s] [ext: omni.kit.mainwindow-1.0.3] startup
[2.399s] [ext: carb.audio-0.1.0] startup
[2.402s] [ext: omni.uiaudio-1.0.0] startup
[2.403s] [ext: omni.kit.uiapp-0.0.0] startup
[2.403s] [ext: omni.usd.schema.audio-0.0.0] startup
[2.464s] [ext: omni.usd.schema.physx-106.0.20] startup
[2.495s] [ext: omni.usd.schema.forcefield-106.0.20] startup
[2.500s] [ext: omni.usd.schema.anim-0.0.0] startup
[2.520s] [ext: omni.usd.schema.omniscripting-1.0.0] startup
[2.525s] [ext: omni.usd.schema.omnigraph-1.0.0] startup
[2.531s] [ext: omni.anim.graph.schema-106.0.2] startup
[2.536s] [ext: omni.anim.navigation.schem

[DEBUG] [AutoNode] Defining data type 'any' as 'Any'
[DEBUG] [AutoNode] Defining data type 'bool' as 'Bool' and array 'BoolArray
[DEBUG] [AutoNode] Defining data type 'bundle' as 'Bundle'
[DEBUG] [AutoNode] Defining data type 'colord[3]' as 'Color3d' and array 'Color3dArray
[DEBUG] [AutoNode] Defining data type 'colorf[3]' as 'Color3f' and array 'Color3fArray
[DEBUG] [AutoNode] Defining data type 'colorh[3]' as 'Color3h' and array 'Color3hArray
[DEBUG] [AutoNode] Defining data type 'colord[4]' as 'Color4d' and array 'Color4dArray
[DEBUG] [AutoNode] Defining data type 'colorf[4]' as 'Color4f' and array 'Color4fArray
[DEBUG] [AutoNode] Defining data type 'colorh[4]' as 'Color4h' and array 'Color4hArray
[DEBUG] [AutoNode] Defining data type 'double' as 'Double' and array 'DoubleArray
[DEBUG] [AutoNode] Defining data type 'double[2]' as 'Double2' and array 'Double2Array
[DEBUG] [AutoNode] Defining data type 'double[3]' as 'Double3' and array 'Double3Array
[DEBUG] [AutoNode] Defining data t

...


                   ____________
                  /          / \
                 /          / /__
                /          / /  /\
               /__________/ /__/  \
               \   _____  \ \__\  /
                \  \  / \  \ \_/ /
                 \  \/___\  \   /
                  \__________\_/  
       ___                  _  ____ _ _                     
      / _ \ _ __ ___  _ __ (_)/ ___(_) |__  ___  ___  _ __  
     | | | | '_ ` _ \| '_ \| | |  _| | '_ \/ __|/ _ \| '_ \ 
     | |_| | | | | | | | | | | |_| | | |_) \__ \ (_) | | | |
      \___/|_| |_| |_|_| |_|_|\____|_|_.__/|___/\___/|_| |_|

2025-09-18 02:42:29 [10,886ms] [Warning] [omni.syntheticdata.plugin] SdRenderVarPtr missing valid input renderVar LdrColorSDhost


[INFO] [omnigibson.simulator] Imported scene 0.
[WARNING] [omnigibson.robots.fetch] Fetch wheel links are post-processed to use sphere approximation collision meshes. Please ignore any previous errors about these collision meshes.
[WARNING] [omnigibson.robots.fetch] Fetch wheel links are post-processed to use sphere approximation collision meshes. Please ignore any previous errors about these collision meshes.


2025-09-18 02:42:32 [14,013ms] [Warning] [omnigibson.robots.fetch] Fetch wheel links are post-processed to use sphere approximation collision meshes. Please ignore any previous errors about these collision meshes.
2025-09-18 02:42:32 [14,013ms] [Warning] [omnigibson.robots.fetch] Fetch wheel links are post-processed to use sphere approximation collision meshes. Please ignore any previous errors about these collision meshes.


/home/youlika/miniconda3/envs/omnigibson/lib/python3.10/site-packages/omnigibson/utils/python_utils.py:730: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  out = th.tensor(nums, dtype=dtype) if isinstance(nums, Iterable) else th.ones(dim, dtype=dtype) * nums
/home/youlika/miniconda3/envs/omnigibson/lib/python3.10/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647352509/work/aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


2025-09-18 02:42:34 [16,028ms] [Warning] [carb] Client omni.kit.viewport.legacy_gizmos has acquired [carb::settings::ISettings v1.0] 100 times. Consider accessing this interface with carb::getCachedInterface() (Performance warning)
2025-09-18 02:42:34 [16,138ms] [Warning] [carb] Client omni.kit.viewport.legacy_gizmos has acquired [carb::dictionary::IDictionary v1.1] 100 times. Consider accessing this interface with carb::getCachedInterface() (Performance warning)
2025-09-18 02:42:34 [16,402ms] [Warning] [omni.syntheticdata.plugin] SdRenderVarToRawArray missing valid input renderVar InstanceSegmentationReductionSD
2025-09-18 02:42:34 [16,402ms] [Warning] [omni.syntheticdata.plugin] OgnSdInstanceMapping missing valid input renderVar InstanceMappingInfoSDhost
Module omni.replicator.core.ogn.python._impl.nodes.OgnSemanticSegmentation cc3f83a load on device 'cuda:0' took 0.21 ms
2025-09-18 02:42:34 [16,494ms] [Warning] [carb] Client omni.hydratexture.plugin has acquired [carb::settings::ISe

In [8]:
init(scene_file, visualize=args.visualize)
disturbance_seq=None
rekep_program_dir=task['rekep_program_dir']

env = o_env
config = CTX['config']

env.reset()

/home/youlika/miniconda3/envs/omnigibson/lib/python3.10/site-packages/gymnasium/spaces/box.py:423: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


Reset done.


In [9]:
action = np.zeros(env.robot.action_dim)
# 给定 base 的前进速度（假设控制器支持）
action[0] = -0.2     # vx

for i in range(40):
    # 执行一步仿真，并打印 step 返回结果（可能是 3 元组或 5 元组，取决于 OmniGibson 版本）
    env.og_env.step(action)

position, orientation = env.robot.get_position_orientation()
print('position:',position)
print('orientation:',orientation)

position: tensor([-1.2835, -0.0471,  0.0031])
orientation: tensor([ 0.0046, -0.0165,  0.0877,  0.9960])


In [10]:
action = np.zeros(env.robot.action_dim)
# 给定 base 的前进速度（假设控制器支持）
action[0] = 0.2     # vx

for i in range(40):
    # 执行一步仿真，并打印 step 返回结果（可能是 3 元组或 5 元组，取决于 OmniGibson 版本）
    env.og_env.step(action)

position, orientation = env.robot.get_position_orientation()
print('position:',position)
print('orientation:',orientation)

position: tensor([-0.8440,  0.0068,  0.0016])
orientation: tensor([ 0.0031, -0.0137,  0.0363,  0.9992])


In [11]:
cam_obs = env.get_cam_obs()
rgb = cam_obs[config['vlm_camera']]['rgb']
points = cam_obs[config['vlm_camera']]['points']
mask = cam_obs[config['vlm_camera']]['seg']

if rekep_program_dir is None:
    keypoints, projected_img = o_KeypointProposer.get_keypoints(rgb, points, mask)
    print(f'{bcolors.HEADER}Got {len(keypoints)} proposed keypoints{bcolors.ENDC}')
    if CTX['visualize']:
        CTX['visualizer'].show_img(projected_img)
    metadata = {'init_keypoint_positions': keypoints, 'num_keypoints': len(keypoints)}
    rekep_program_dir = o_constraint_generator.generate(projected_img, instruction, metadata)
    print(f'{bcolors.HEADER}Constraints generated{bcolors.ENDC}')

###### 下面是 execute 部分
with open(os.path.join(rekep_program_dir, 'metadata.json'), 'r') as f:
    CTX['program_info'] = json.load(f)

CTX['applied_disturbance'] = {stage: False for stage in range(1, CTX['program_info']['num_stages'] + 1)}
env.register_keypoints(CTX['program_info']['init_keypoint_positions'])

CTX['constraint_fns'] = {}
for stage in range(1, CTX['program_info']['num_stages'] + 1):
    stage_dict = {}
    for constraint_type in ['subgoal', 'path']:
        load_path = os.path.join(rekep_program_dir, f'stage{stage}_{constraint_type}_constraints.txt')
        get_grasping_cost_fn = get_callable_grasping_cost_fn(env)
        stage_dict[constraint_type] = load_functions_from_txt(load_path, get_grasping_cost_fn) if os.path.exists(load_path) else []
    CTX['constraint_fns'][stage] = stage_dict


running k-means on cuda..


[running kmeans]: 6it [00:00, 27.50it/s, center_shift=0.000052, iteration=6, tol=0.000100]


running k-means on cuda..


[running kmeans]: 14it [00:00, 263.59it/s, center_shift=0.000063, iteration=14, tol=0.000100]


running k-means on cuda..


[running kmeans]: 30it [00:00, 398.02it/s, center_shift=0.000071, iteration=30, tol=0.000100]


running k-means on cuda..


[running kmeans]: 6it [00:00, 655.91it/s, center_shift=0.000053, iteration=6, tol=0.000100]
/home/youlika/miniconda3/envs/omnigibson/lib/python3.10/site-packages/sklearn/cluster/_mean_shift.py:291: UserWarning: Binning data failed with provided bin_size=0.060000, using data points as seeds.
  warnings.warn(


Got 9 proposed keypoints
保存目录： /home/youlika/code/github/Rekep/./vlm_query/2025-09-18_10-44-02_reorient_the_red_pen_and_drop_it_upright_into_the_black_pen_holder
Constraints saved to /home/youlika/code/github/Rekep/./vlm_query/2025-09-18_10-44-02_reorient_the_red_pen_and_drop_it_upright_into_the_black_pen_holder
Metadata saved to /home/youlika/code/github/Rekep/./vlm_query/2025-09-18_10-44-02_reorient_the_red_pen_and_drop_it_upright_into_the_black_pen_holder/metadata.json
Constraints generated


In [12]:
CTX['constraint_fns']

{1: {'subgoal': [<function stage1_subgoal_constraint1(end_effector, keypoints)>],
  'path': []},
 2: {'subgoal': [<function stage2_subgoal_constraint1(end_effector, keypoints)>],
  'path': [<function stage2_path_constraint1(end_effector, keypoints)>]},
 3: {'subgoal': [<function stage3_subgoal_constraint1(end_effector, keypoints)>,
   <function stage3_subgoal_constraint2(end_effector, keypoints)>],
  'path': [<function stage3_path_constraint1(end_effector, keypoints)>]}}

: 

In [12]:
# === 主控制循环：感知 → 规划 → 执行 → 阶段推进/回溯 ===
# 流程说明：
# - 初始化关键点可动性掩码与阶段，从阶段1开始。
# - 每一轮：
#   1) 从环境读取当前末端位姿、关节角、关键点位置、碰撞与SDF体素。
#   2) 若非首阶段，检查当前阶段路径约束是否被违反；若违反，则回溯到最近满足路径约束的阶段。
#   3) 若无需回溯：根据当前阶段的子目标/路径约束调用求解器，生成子目标与路径，并转换为稠密动作序列，逐步执行。
#   4) 阶段末尾：若需抓取/释放则执行相应动作；若到达最后阶段则保存视频并退出，否则推进到下一阶段。
# 关键变量：
# - CTX['constraint_fns']：每阶段subgoal/path约束函数集合
# - CTX['is_grasp_stage'] / CTX['is_release_stage']：阶段类型开关
# - CTX['action_queue']：待执行的末端轨迹动作序列
# - CTX['first_iter']：阶段中的首轮优化，通常从头求解以获得更稳定的初解

CTX['keypoint_movable_mask'] = np.zeros(CTX['program_info']['num_keypoints'] + 1, dtype=bool)
CTX['keypoint_movable_mask'][0] = True

CTX['last_sim_step_counter'] = -np.inf
update_stage(1)

program_is_over = False
while True:
    scene_keypoints = env.get_keypoint_positions()
    CTX['keypoints'] = np.concatenate([[env.get_ee_pos()], scene_keypoints], axis=0)
    CTX['curr_ee_pose'] = env.get_ee_pose()
    CTX['curr_joint_pos'] = env.get_arm_joint_postions()
    CTX['sdf_voxels'] = env.get_sdf_voxels(CTX['config']['sdf_voxel_size'])
    CTX['collision_points'] = env.get_collision_points()

    # 回溯判断
    backtrack = False
    if CTX['stage'] > 1:
        path_constraints = CTX['constraint_fns'][CTX['stage']]['path']
        for constraints in path_constraints:
            violation = constraints(CTX['keypoints'][0], CTX['keypoints'][1:])
            if violation > CTX['config']['constraint_tolerance']:
                backtrack = True
                break
    if backtrack:
        for new_stage in range(CTX['stage'] - 1, 0, -1):
            path_constraints = CTX['constraint_fns'][new_stage]['path']
            if len(path_constraints) == 0:
                break
            all_constraints_satisfied = True
            for constraints in path_constraints:
                violation = constraints(CTX['keypoints'][0], CTX['keypoints'][1:])
                if violation > CTX['config']['constraint_tolerance']:
                    all_constraints_satisfied = False
                    break
            if all_constraints_satisfied:
                break
        print(f"{bcolors.HEADER}[stage={CTX['stage']}] backtrack to stage {new_stage}{bcolors.ENDC}")
        update_stage(new_stage)
    else:
        update_disturbance_seq(CTX['stage'], disturbance_seq)

        if CTX['last_sim_step_counter'] == env.step_counter:
            print(f"{bcolors.WARNING}sim did not step forward...{bcolors.ENDC}")

        next_subgoal = get_next_subgoal(from_scratch=CTX['first_iter'])
        next_path = get_next_path(next_subgoal, from_scratch=CTX['first_iter'])
        CTX['first_iter'] = False
        CTX['action_queue'] = next_path.tolist()
        CTX['last_sim_step_counter'] = env.step_counter

        count = 0
        while len(CTX['action_queue']) > 0 and count < CTX['config']['action_steps_per_iter']:
            next_action = CTX['action_queue'].pop(0)
            precise = len(CTX['action_queue']) == 0
            env.execute_action(next_action, precise=precise)
            count += 1
        if len(CTX['action_queue']) == 0:
            if CTX['is_grasp_stage']:
                execute_grasp_action()
            elif CTX['is_release_stage']:
                execute_release_action()

            # 执行完动作序列后进行阶段判断
            if CTX['stage'] == CTX['program_info']['num_stages']: # num_stages 是阶段总数
                env.sleep(2.0)
                save_path = env.save_video()
                print(f"{bcolors.OKGREEN}Video saved to {save_path}\n\n{bcolors.ENDC}")
                print("程序正常运行结束")
                program_is_over = True
                break
            else:   
                update_stage(CTX['stage'] + 1)

        if CTX['stage'] == CTX['program_info']['num_stages'] and program_is_over:
                break # 跳出外层循环



########################################
# Optimization debug info:
# collision_cost         : 0.60993
# init_pose_cost         : 0.30496
# ik_feasible            : 1.00000
# ik_pos_error           : 0.00000
# ik_cost                : 1.00000
# reset_reg_cost         : 0.17910
# grasp_cost             : 0.00027
# subgoal_constraint_cost: 0.00034
# subgoal_violation      : [0.]
# path_violation         : None
# total_cost             : 2.09459
# sol                    : [ 0.12305214 -0.15704369 -0.93223865 -0.27147121  0.50232453 -0.72924555]
# msg                    : Maximum number of function call reached during annealing
# solve_time             : 1.72573
# from_scratch           : 1.00000
# type                   : subgoal_solver
# stage                  : 1.00000
########################################


########################################
# Optimization debug info:
# num_control_points: 1.00000
# num_poses         : 14.00000
# collision_cost    : 3.99372
# path_length_cost

[WARNING] [omnigibson.utils.usd_utils] Contact data for prim /World/scene_0/pen_2/base_link and /World/scene_0/controllable__fetch__Fetch/l_gripper_finger_link is missing because there are too many contacts!


2025-09-18 02:44:55 [157,586ms] [Warning] [omni.physx.tensors.plugin] Incomplete contact data is reported in CpuRigidContactView::getContactData because there are more contact data points than specified maxContactDataCount = 4.
2025-09-18 02:44:55 [157,587ms] [Warning] [omnigibson.utils.usd_utils] Contact data for prim /World/scene_0/pen_2/base_link and /World/scene_0/controllable__fetch__Fetch/l_gripper_finger_link is missing because there are too many contacts!

########################################
# Optimization debug info:
# collision_cost         : 0.04164
# init_pose_cost         : 2.73375
# ik_feasible            : 1.00000
# ik_pos_error           : 0.00000
# ik_cost                : 1.00000
# reset_reg_cost         : 0.32985
# subgoal_constraint_cost: 0.00080
# subgoal_violation      : [0.]
# path_violation         : [0]
# path_constraint_cost   : 0.00000
# total_cost             : 4.10604
# sol                    : [ 0.22765849 -0.27164156 -0.02673887  0.0186399  -0.019958

In [11]:
rekep_program_dir

'./vlm_query/pen'

: 